In [ ]:
# BACKUP
# Fungsi untuk mulai clone repo mahasiswa dan menyisakan file .py dan .ipynb

def clone_and_filter_repo(repo_url, target_dir, delay=2):
    global success_count, failed_count, skipped_count
    
    repo_name = extract_repo_name(repo_url)

    # Amankan nama folder dari karakter ilegal Windows
    folder_name = re.sub(r'[<>:"/\\|?* ]+', '_', repo_name)

    target_path = os.path.join(target_dir, folder_name)

    if os.path.exists(target_path):
        print(f"[SKIP] {folder_name} sudah ada.")
        write_log(folder_name, repo_url, "SKIPPED", "Folder sudah ada")
        skipped_count += 1
        return

    print(f"[CLONE] {folder_name}")

    try:
        subprocess.run(
            ["git", "clone", repo_url, target_path],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        # Seleksi file kode untuk menghapus file selain .py/.ipynb
        keep_only_python_files(target_path)

        write_log(folder_name, repo_url, "SUCCESS", "Clone berhasil")
        success_count += 1

        time.sleep(delay)

    except subprocess.TimeoutExpired:
        print(f"--- (FAILED)Timeout saat clone {repo_url}")
        write_log(folder_name, repo_url, "FAILED", "Timeout")
        failed_count += 1

    except subprocess.CalledProcessError as e:
        print(f"--- (FAILED)Gagal clone {repo_url} (Cek izin repo/token)---")
        print(e.stderr.decode())
        error_msg = e.stderr.decode(errors="ignore")
        write_log(folder_name, repo_url, "FAILED", error_msg)
        failed_count += 1


In [ ]:
# NORMALISASI DIREKTORI MAHASISWA

def normalize_student_directory(student_path):

    student_name = os.path.basename(student_path)

    print(f"\n🔎 Cek struktur mahasiswa: {student_name}")

    # VALIDASI STRUKTUR
    if not is_valid_module_structure(student_path):
        print(f"⚠ STRUKTUR TIDAK SESUAI → {student_name} DI-SKIP")
        return

    print(f"✅ Struktur valid → mulai normalisasi {student_name}")

    for item in os.listdir(student_path):

        old_path = os.path.join(student_path, item)

        if not os.path.isdir(old_path):
            continue

        new_name = normalize_module_folder_name(item)

        if new_name is None:
            continue

        new_path = os.path.join(student_path, new_name)

        if old_path != new_path:
            print(f"   🔄 Folder: {item} → {new_name}")
            os.rename(old_path, new_path)
        else:
            print(f"   ✔ Folder: {item} (tidak berubah)")

        print(f"      📄 Normalisasi file di folder {new_name}")
        normalize_task_files(new_path)

    print(f"✅ Selesai normalisasi mahasiswa: {student_name}")


## seluruh project (cprofile) versi 1

In [ ]:

# KONFIGURASI FOLDER
dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(10)_py"

output_csv = os.path.join(log_folder, "runtime_execution.csv")
output_excel = os.path.join(log_folder, "runtime_execution.xlsx")
output_json = os.path.join(log_folder, "runtime_execution.json")
# RUN FILE DENGAN CPROFILE

def run_python_file(file_path):

    try:

        start = time.perf_counter()

        subprocess.run(
            ["python", "-m", "cProfile", file_path],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            timeout=30
        )

        end = time.perf_counter()

        runtime = end - start

        return "SUCCESS", runtime, ""

    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, "Execution timeout"

    except Exception as e:
        return "FAILED", None, str(e)
# EKSTRAK METADATA FILE

def extract_metadata(file_path, dataset_folder):

    # ambil path relatif dari dataset
    rel_path = os.path.relpath(file_path, dataset_folder)

    parts = rel_path.split(os.sep)

    # file name
    file_name = parts[-1]

    # nim = folder pertama
    nim = parts[0] if len(parts) >= 2 else "unknown"

    # module = folder setelah nim (jika ada)
    module = parts[1] if len(parts) >= 3 else "root"

    return nim, module, file_name
# LOOP DATASET

records = []
total_files = 0

print("\n========== MULAI EKSEKUSI FILE ==========\n")

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        total_files += 1

        nim, module, file_name = extract_metadata(file_path, dataset_folder)

        print(f"[RUN] {nim}/{module}/{file_name}")

        status, runtime, message = run_python_file(file_path)

        records.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "nim": nim,
            "module": module,
            "file_name": file_name,
            "file_path": file_path,
            "status": status,
            "execution_time_seconds": runtime,
            "message": message
        })

print("\n========== EKSEKUSI SELESAI ==========\n")
# DATAFRAME
df = pd.DataFrame(records)

# format runtime
df["execution_time_seconds"] = df["execution_time_seconds"].round(6)

# SIMPAN Hasil
df.to_csv(output_csv, index=False)
df.to_excel(output_excel, index=False)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4)

# TAMPILKAN TABEL 

print("========== HASIL RUNTIME ==========\n")

display_columns = [
    "nim",
    "module",
    "file_name",
    "status",
    "execution_time_seconds"
]

display(df[display_columns])
# STATISTIK RINGKAS

print("\n========== RINGKASAN ==========\n")
summary = {
    "Total file dijalankan": total_files,
    "SUCCESS": (df["status"] == "SUCCESS").sum(),
    "FAILED": (df["status"] == "FAILED").sum(),
    "TIMEOUT": (df["status"] == "TIMEOUT").sum(),
    "Rata-rata runtime": round(df["execution_time_seconds"].mean(), 6)
}

summary_df = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])

display(summary_df)

print("\nTop 5 runtime terlama:")

top_runtime = (
    df.sort_values("execution_time_seconds", ascending=False)
    [["nim","module","file_name","execution_time_seconds"]]
    .head()
)

display(top_runtime)

print("\nHasil lengkap disimpan di:")
print(output_csv)
print(output_excel)
print(output_json)

## per function (timeit)

In [ ]:
import os
import ast
import timeit
import pandas as pd
import contextlib
import io
import builtins
from datetime import datetime

# =====================================
# KONFIGURASI
# =====================================

dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(5)_py"

output_file = os.path.join(log_folder, "runtime_function_timeit.csv")

RUN_REPEAT = 5


# =====================================
# EKSTRAK METADATA
# =====================================

def extract_metadata(file_path):

    parts = file_path.split(os.sep)

    try:
        nim = parts[-3]
        module = parts[-2]
        file_name = parts[-1]
    except:
        nim = "unknown"
        module = "unknown"
        file_name = os.path.basename(file_path)

    return nim, module, file_name


# =====================================
# AMBIL FUNCTION DARI AST
# =====================================

def get_functions(file_path):

    with open(file_path, "r", encoding="utf-8") as f:
        source = f.read()

    tree = ast.parse(source)

    functions = []

    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            functions.append(node)

    return functions


# =====================================
# UKUR RUNTIME FUNCTION
# =====================================

def measure_function_runtime(function_node):

    try:

        module = ast.Module(body=[function_node], type_ignores=[])

        compiled = compile(module, filename="<ast>", mode="exec")

        namespace = {}

        exec(compiled, namespace)

        func = namespace[function_node.name]

        def wrapper():

            # blokir print mahasiswa
            f = io.StringIO()

            with contextlib.redirect_stdout(f):

                # override input
                original_input = builtins.input
                builtins.input = lambda *args: "0"

                try:
                    func()
                finally:
                    builtins.input = original_input

        runtime = timeit.timeit(wrapper, number=RUN_REPEAT)

        avg_runtime = runtime / RUN_REPEAT

        return "SUCCESS", avg_runtime, ""

    except Exception as e:

        return "FAILED", None, str(e)


# =====================================
# LOOP DATASET
# =====================================

records = []

print("\n========== MULAI ANALISIS FUNCTION RUNTIME ==========\n")

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        nim, module, file_name = extract_metadata(file_path)

        try:
            functions = get_functions(file_path)

        except Exception as e:

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": "parse_error",
                "status": "FAILED",
                "runtime_seconds": None,
                "message": str(e)
            })

            continue

        for func in functions:

            print(f"[RUN] {nim}/{module}/{file_name} → {func.name}")

            status, runtime, message = measure_function_runtime(func)

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": func.name,
                "status": status,
                "runtime_seconds": runtime,
                "message": message
            })


print("\n========== ANALISIS SELESAI ==========\n")


# =====================================
# DATAFRAME
# =====================================

df = pd.DataFrame(records)

df["runtime_seconds"] = df["runtime_seconds"].round(8)


# =====================================
# SIMPAN HASIL
# =====================================

df.to_csv(output_file, index=False)

print("Hasil runtime disimpan di:")
print(output_file)


# =====================================
# TABEL OUTPUT
# =====================================

display_cols = [
    "nim",
    "module",
    "file_name",
    "function_name",
    "status",
    "runtime_seconds"
]

print("\n========== TABEL HASIL ==========\n")

print(df[display_cols].to_string(index=False))


# =====================================
# STATISTIK
# =====================================

print("\n========== RINGKASAN ==========\n")

print("Total function dianalisis:", len(df))

print("\nStatus eksekusi:")
print(df["status"].value_counts())

print("\nTop 10 runtime paling lama:")

print(
    df.sort_values("runtime_seconds", ascending=False)
    [["nim","module","file_name","function_name","runtime_seconds"]]
    .head(10)
    .to_string(index=False)
)


========== MULAI ANALISIS FUNCTION RUNTIME ==========

[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → train_xor
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P2_JS13.py → build_model
[RUN] 2241720092/Klasterisasi/P2_JS04.py → find_clusters
[RUN] 2241720092/Klasterisasi/P2_JS04.py → plot_pixels
[RUN] 2341720095/JS04/P2_JS04.py → find_cluster

TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'

## seluruh project (cprofile) versi 1

In [ ]:
# KONFIGURASI FOLDER
dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(5)_py"

output_file = os.path.join(log_folder, "runtime_execution.csv")

In [ ]:
# RUN FILE DENGAN CPROFILE

def run_python_file(file_path):

    try:

        start = time.perf_counter()

        subprocess.run(
            ["python", "-m", "cProfile", file_path],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            timeout=30
        )

        end = time.perf_counter()

        runtime = end - start

        return "SUCCESS", runtime, ""

    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, "Execution timeout"

    except Exception as e:
        return "FAILED", None, str(e)

In [ ]:
# EKSTRAK METADATA FILE

def extract_metadata(file_path):

    parts = file_path.split(os.sep)

    try:
        nim = parts[-3]
        module = parts[-2]
        file_name = parts[-1]
    except:
        nim = "unknown"
        module = "unknown"
        file_name = os.path.basename(file_path)

    return nim, module, file_name

In [ ]:
# LOOP DATASET

records = []
total_files = 0

print("\n========== MULAI EKSEKUSI FILE ==========\n")

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        total_files += 1

        nim, module, file_name = extract_metadata(file_path)

        print(f"[RUN] {nim}/{module}/{file_name}")

        status, runtime, message = run_python_file(file_path)

        records.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "nim": nim,
            "module": module,
            "file_name": file_name,
            "file_path": file_path,
            "status": status,
            "execution_time_seconds": runtime,
            "message": message
        })

print("\n========== EKSEKUSI SELESAI ==========\n")


========== MULAI EKSEKUSI FILE ==========

[RUN] kode_github(5)_py/2241720092/Uts.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P2_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P3_JS13.py
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/TP_JS13.py
[RUN] 2241720092/Convolutional Neural Network/P1_JS14.py
[RUN] 2241720092/Convolutional Neural Network/P2_JS14.py
[RUN] 2241720092/Convolutional Neural Network/TP_JS14.py
[RUN] 2241720092/Ekstraksi Fitur/P1_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P2_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P3_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/P4_JS03.py
[RUN] 2241720092/Ekstraksi Fitur/TP_JS03.py
[RUN] 2241720092/Klasterisasi/P1_JS04.py
[RUN] 2241720092/Klasterisasi/P2_JS04.py
[RUN] 2241720092/Klasterisasi/P3_JS04.py
[RUN] 2241720092/Klasterisasi/TP_JS04.py
[RUN] 2241720092/Pemaha

In [ ]:
# DATAFRAME
df = pd.DataFrame(records)

# format runtime
df["execution_time_seconds"] = df["execution_time_seconds"].round(6)

# SIMPAN CSV
df.to_csv(output_file, index=False)

# TAMPILKAN TABEL 

print("========== HASIL RUNTIME ==========\n")

display_columns = [
    "nim",
    "module",
    "file_name",
    "status",
    "execution_time_seconds"
]

print(df[display_columns].to_string(index=False))

========== HASIL RUNTIME ==========

               nim                                                  module                                        file_name  status  execution_time_seconds
 kode_github(5)_py                                              2241720092                                           Uts.py SUCCESS                2.227066
        2241720092 Artificial Neural Network (ANN) dan Evaluasi Classifier                                       P1_JS13.py SUCCESS                0.795189
        2241720092 Artificial Neural Network (ANN) dan Evaluasi Classifier                                       P2_JS13.py SUCCESS                1.962561
        2241720092 Artificial Neural Network (ANN) dan Evaluasi Classifier                                       P3_JS13.py SUCCESS                1.867639
        2241720092 Artificial Neural Network (ANN) dan Evaluasi Classifier                                       TP_JS13.py SUCCESS                0.111810
        2241720092         

In [ ]:
# STATISTIK RINGKAS

print("\n========== RINGKASAN ==========\n")

print(f"Total file dijalankan : {total_files}\n")

print("Status execution:")
print(df["status"].value_counts())

print("\nRata-rata runtime (detik):")
print(df["execution_time_seconds"].mean())

print("\nTop 5 runtime terlama:")

print(
    df.sort_values("execution_time_seconds", ascending=False)
    [["nim","module","file_name","execution_time_seconds"]]
    .head()
    .to_string(index=False)
)

print("\nHasil lengkap disimpan di:")
print(output_file)


========== RINGKASAN ==========

Total file dijalankan : 170

Status execution:
SUCCESS    169
TIMEOUT      1
Name: status, dtype: int64

Rata-rata runtime (detik):
0.8501759467455621

Top 5 runtime terlama:
       nim module  file_name  execution_time_seconds
2341720217   JS13 P1_JS13.py                2.473560
2341720095   JS11 P4_JS11.py                2.373491
2341720217   JS13 P2_JS13.py                2.296521
2341720217   JS11 P4_JS11.py                2.277760
2341720095   JS03 P2_JS03.py                2.256220

Hasil lengkap disimpan di:
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\logPreprocessing\runtime_execution.csv


In [ ]:
import os
import ast
import timeit
import pandas as pd
import contextlib
import io
import builtins
import threading
from datetime import datetime

# =====================================
# KONFIGURASI
# =====================================

dataset_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_github(5)_py"

output_file = os.path.join(log_folder, "runtime_per_function2.csv")

RUN_REPEAT = 3
TIMEOUT = 5


# =====================================
# BLOK INPUT
# =====================================

builtins.input = lambda *args: "0"


# =====================================
# METADATA
# =====================================

def extract_metadata(file_path):

    parts = file_path.split(os.sep)

    try:
        nim = parts[-3]
        module = parts[-2]
        file_name = parts[-1]
    except:
        nim = "unknown"
        module = "unknown"
        file_name = os.path.basename(file_path)

    return nim, module, file_name


# =====================================
# EXTRACT FUNCTION DARI AST
# =====================================

def extract_functions(file_path):

    with open(file_path, "r", encoding="utf8", errors="ignore") as f:
        code = f.read()

    try:
        tree = ast.parse(code)
    except:
        return []

    functions = []

    for node in tree.body:

        if isinstance(node, ast.FunctionDef):

            func_code = ast.get_source_segment(code, node)

            functions.append((node.name, func_code))

    return functions


# =====================================
# GENERATE WRAPPER
# =====================================

def generate_wrapper(func_name, func_code):

    try:

        tree = ast.parse(func_code)
        func_node = tree.body[0]

        param_count = len(func_node.args.args)

        dummy_args = ",".join(["0"] * param_count)

    except:

        dummy_args = ""

    wrapper = f"""
{func_code}

def __wrapper():
    try:
        {func_name}({dummy_args})
    except:
        pass
"""

    return wrapper


# =====================================
# RUN DENGAN TIMEOUT (THREAD)
# =====================================

def run_with_timeout(code):

    result = {"status":None,"runtime":None,"message":""}

    def worker():

        try:

            local_env = {}

            dummy = io.StringIO()

            with contextlib.redirect_stdout(dummy), contextlib.redirect_stderr(dummy):

                exec(code, local_env)

                runtime = timeit.timeit(
                    "__wrapper()",
                    globals=local_env,
                    number=RUN_REPEAT
                )

            result["status"] = "SUCCESS"
            result["runtime"] = runtime / RUN_REPEAT

        except Exception as e:

            result["status"] = "FAILED"
            result["message"] = str(e)

    thread = threading.Thread(target=worker)

    thread.start()
    thread.join(TIMEOUT)

    if thread.is_alive():

        return "TIMEOUT", None, "Execution timeout"

    return result["status"], result["runtime"], result["message"]


# =====================================
# ANALISIS DATASET
# =====================================

records = []

print("\n===== MULAI ANALISIS RUNTIME =====\n")

skip_funcs = ["main","menu","run","start","program"]

for root, dirs, files in os.walk(dataset_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        nim, module, file_name = extract_metadata(file_path)

        functions = extract_functions(file_path)

        if not functions:

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": "NO_FUNCTION",
                "status": "FAILED",
                "runtime_seconds": None,
                "message": "No function detected"
            })

            continue

        for func_name, func_code in functions:

            if func_name.lower() in skip_funcs:
                continue

            print(f"[RUN] {nim}/{module}/{file_name} → {func_name}")

            wrapper_code = generate_wrapper(func_name, func_code)

            status, runtime, message = run_with_timeout(wrapper_code)

            records.append({
                "timestamp": datetime.now(),
                "nim": nim,
                "module": module,
                "file_name": file_name,
                "function_name": func_name,
                "status": status,
                "runtime_seconds": runtime,
                "message": message
            })


print("\n===== ANALISIS SELESAI =====\n")


# =====================================
# SIMPAN CSV
# =====================================

df = pd.DataFrame(records)

df["runtime_seconds"] = pd.to_numeric(df["runtime_seconds"], errors="coerce")

df.to_csv(output_file, index=False)

print("Hasil disimpan di:", output_file)


# =====================================
# RINGKASAN
# =====================================

print("\n===== RINGKASAN =====\n")

print("Total function dianalisis :", len(df))

print("\nStatus eksekusi:")
print(df["status"].value_counts())

print("\nTop runtime:\n")

print(
    df.sort_values("runtime_seconds", ascending=False)
    [["nim","module","file_name","function_name","runtime_seconds"]]
    .head(10)
)


===== MULAI ANALISIS RUNTIME =====

[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → sigmoid_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → relu_derivative
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P1_JS13.py → train_xor
[RUN] 2241720092/Artificial Neural Network (ANN) dan Evaluasi Classifier/P2_JS13.py → build_model
[RUN] 2241720092/Klasterisasi/P2_JS04.py → find_clusters
[RUN] 2241720092/Klasterisasi/P2_JS04.py → plot_pixels
[RUN] 2341720095/JS04/P2_JS04.py → find_clusters
[RUN] 2341720095/

# pengumpulan data

In [ ]:
# CLONE FUNCTION

results = []


def clone_repo(nim, url):

    global success_count, fail_count, skip_count
    global total_py_files, total_ipynb_files

    # Jika NIM kosong
    if not nim:
        print(f"[SKIP] URL tanpa NIM → {url}")
        write_log("", url, "SKIPPED", "NIM kosong")
        skip_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "SKIPPED",
            "message": "NIM Kosong",
            "module": 0,
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })
        return

    # Bersihkan URL jika ada /tree/
    if "/tree/" in url:
        url = url.split("/tree/")[0]

    target_path = os.path.join(projects_folder, nim)

    # Jika sudah pernah clone
    if os.path.exists(target_path):
        print(f"[SKIP] {nim} sudah ada.")
        write_log(nim, url, "SKIPPED", "Folder sudah ada")
        skip_count += 1

        py_count, ipynb_count = count_code_files(target_path)
        module_count = count_modules(target_path)

        results.append({
            "nim": nim,
            "url": url,
            "status": "SKIPPED",
            "message": "Folder sudah ada",
            "module": module_count,
            "py_files": py_count,
            "ipynb_files": ipynb_count,
            "total_files": (py_count + ipynb_count)
        })

        return

    print(f"[CLONE] {url}")

    try:

        auth_url = add_token(url)

        subprocess.run(
            ["git", "clone", "--depth", "1", auth_url, target_path],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        # subprocess.run(
        #     ["git", "clone", url, target_path],
        #     check=True,
        #     timeout=300,
        #     stdout=subprocess.DEVNULL,
        #     stderr=subprocess.PIPE
        # )

        # Simpan hanya .py dan .ipynb
        keep_only_code_files(target_path)

        # Hitung jumlah file kode
        py_count, ipynb_count = count_code_files(target_path)
        module_count = count_modules(target_path)

        total_py_files += py_count
        total_ipynb_files += ipynb_count

        print(
            f"   ✅ Berhasil clone {nim}  ||      Jumlah file dalam repo .py: {py_count} | .ipynb: {ipynb_count}")

        write_log(nim, url, "SUCCESS",
                  f"Clone berhasil | module:{module_count} | py:{py_count} ipynb:{ipynb_count}")

        success_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "SUCCESS",
            "message": "Clone berhasil",
            "module": module_count,
            "py_files": py_count,
            "ipynb_files": ipynb_count,
            "total_files": py_count + ipynb_count
        })

        time.sleep(1)

    except subprocess.CalledProcessError as e:

        error_msg = e.stderr.decode(errors="ignore")

        print(f"   ❌ Gagal clone {nim}")
        print(f"   Alasan:\n{error_msg}")

        write_log(nim, url, "FAILED", error_msg)

        # Hapus folder jika setengah clone
        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path, onerror=remove_readonly)
                print("   🧹 Folder clone dihapus")
            except Exception as delete_error:
                print("   ⚠ Gagal hapus folder clone")
                print(str(delete_error))

        fail_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "FAILED",
            "message": error_msg,
            "module": 0,
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })

    except subprocess.TimeoutExpired:

        print(f"   ⏰ Timeout clone {nim}")
        write_log(nim, url, "FAILED", "Timeout")

        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path)
            except:
                pass

        fail_count += 1

        results.append({
            "nim": nim,
            "url": url,
            "status": "FAILED",
            "message": "Clone Timeout",
            "module": 0,
            "py_files": 0,
            "ipynb_files": 0,
            "total_files": 0
        })

In [ ]:
#baru
normalized_folder = projects_folder + "_normalized"
os.makedirs(normalized_folder, exist_ok=True)


# =====================================================
# IDENTIFIKASI MODULE
# =====================================================

def identify_module(text):

    text = text.upper()

    # UTS
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", text):
        return "uts"

    # UAS
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", text):
        return "uas"
    
    # PBL
    if "PBL" in text:
        return "pbl"
    
    # KUIS
    if re.search(r"KUIS|QUIZ", text):
        return "kuis"
    
    # KELOMPOK
    match = re.search(r"KELOMPOK|GROUP", text)
    if match:
        return "kelompok"
    
    # JS / JOBSHEET / PERTEMUAN
    match = re.search(r"(JS|JOBSHEET|PERTEMUAN|SESI|MODUL|MODULE)\s*0?(\d+)", text)
    if match:
        return f"js{int(match.group(2)):02d}"

    # PRAKTIKUM
    match = re.search(r"PRAKTIKUM\s*0?(\d+)", text)
    if match:
        return f"js{int(match.group(1)):02d}"

    return None


# =====================================================
# IDENTIFIKASI FILE TYPE
# =====================================================

def normalize_filename(file):

    name = file.upper()
    ext = os.path.splitext(file)[1].lower()
    
    # PBL
    if "PBL" in name:
        return f"pbl{ext}"
    
    # KELOMPOK
    if re.search(r"KELOMPOK|GROUP", name):
        return f"kelompok{ext}"

    # UTS
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", name):
        return f"uts{ext}"

    # UAS
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", name):
        return f"uas{ext}"
    
    # KUIS
    if re.search(r"KUIS|QUIZ", name):
        return f"kuis{ext}"

    # PRAKTIKUM
    match = re.search(r"\bP\s*0?(\d+)", name)
    if match:
        return f"p{int(match.group(1)):02d}{ext}"

    # TUGAS PRAKTIKUM
    match = re.search(r"(TP|TG|TUGAS)\s*0?(\d+)", name)
    if match:
        return f"tp{int(match.group(2)):02d}{ext}"

    return None


# =====================================================
# HAPUS FOLDER YANG TIDAK PERLU
# =====================================================

def remove_unnecessary(repo, remove_list, nim):

    for root, dirs, files in os.walk(repo):

        for d in dirs[:]:
            if d in remove_list:
                path = os.path.join(root, d)

                try:
                    shutil.rmtree(path)
                    dirs.remove(d)
                    write_log(nim, "REMOVE_FOLDER", path)
                except Exception as e:
                    write_log(nim, "REMOVE_FOLDER", path, "FAILED", str(e))


# =====================================================
# CARI MODULE DARI PATH
# =====================================================

def detect_module_from_path(root, file):

    module = None

    # cek semua folder dalam path
    for part in root.split(os.sep):

        module = identify_module(part)

        if module:
            return module

    # jika tidak ditemukan → cek dari file
    module = identify_module(file)

    return module


# =====================================================
# NORMALISASI REPOSITORY
# =====================================================

def normalize_repository(repo_path):

    nim = os.path.basename(repo_path)

    print(f"Normalisasi {nim}")

    normalized_repo = os.path.join(normalized_folder, nim)

    if os.path.exists(normalized_repo):
        shutil.rmtree(normalized_repo)

    shutil.copytree(repo_path, normalized_repo)

    remove_list = [".env", ".venv", "env", "venv", "__pycache__", ".git"]
    remove_unnecessary(normalized_repo, remove_list, nim)

    unclassified = os.path.join(normalized_repo, "unclassified")
    os.makedirs(unclassified, exist_ok=True)

    moved_files = []

    # scan semua file dulu
    for root, dirs, files in os.walk(normalized_repo):
        # jangan masuk ke folder .git
        dirs[:] = [d for d in dirs if d != ".git"]
        for file in files:

            if not (file.endswith(".py") or file.endswith(".ipynb")):
                continue

            file_path = os.path.join(root, file)

            moved_files.append((root, file, file_path))


    # proses pemindahan
    for root, file, file_path in moved_files:

        module = detect_module_from_path(root, file)

        if module:
            target_folder = os.path.join(normalized_repo, module)
        else:
            target_folder = unclassified

        os.makedirs(target_folder, exist_ok=True)

        new_name = normalize_filename(file)
        if new_name and new_name != file:
            write_log(nim, "RENAME_FILE", f"{file} -> {new_name}")

        if not new_name:
            new_name = file

        target_path = os.path.join(target_folder, new_name)

        # hindari overwrite
        counter = 1
        base, ext = os.path.splitext(new_name)

        while os.path.exists(target_path):

            target_path = os.path.join(
                target_folder,
                f"{base}_{counter}{ext}"
            )
            write_log(nim, "DUPLICATE_RENAME", f"{new_name} -> {base}_{counter}{ext}")
            counter += 1

        try:
            shutil.move(file_path, target_path)
            write_log(nim, "MOVE_FILE", f"{file_path} -> {target_path}")
        except Exception as e:
            write_log(nim, "MOVE_FILE", file_path, "FAILED", str(e))

    print("   selesai")


# =====================================================
# NORMALISASI DATASET
# =====================================================

print("\nMulai normalisasi dataset...\n")

total_repo = 0

for repo in os.listdir(projects_folder):

    repo_path = os.path.join(projects_folder, repo)

    if os.path.isdir(repo_path):
        # print(repo_path)
        # normalize_repository(repo_path)
        total_repo += 1


print("\n===== NORMALISASI SELESAI =====")
print(f"Total repository : {total_repo}")

# ===============================
# SIMPAN LOG
# ===============================

df_log = pd.DataFrame(log_records)

df_log.to_excel(normStruktur_excel, index=False)

print("\nLog normalisasi disimpan di:")
print(normStruktur_excel)

print("="*35)
print("DATA LOG NORMALISASI")
print("-"*35)
display(df_log.head())

# normalisasi

In [ ]:
#backup normalisasi baru

normalized_folder = projects_folder + "_normalized"
os.makedirs(normalized_folder, exist_ok=True)


# =====================================================
# IDENTIFIKASI MODULE
# =====================================================

def identify_module(text):

    text = text.upper()

    # UTS
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", text):
        return "uts"

    # UAS
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", text):
        return "uas"
    
    # PBL
    if "PBL" in text:
        return "pbl"
    
    # KUIS
    if re.search(r"KUIS|QUIZ", text):
        return "kuis"
    
    # KELOMPOK
    match = re.search(r"KELOMPOK|GROUP", text)
    if match:
        return "kelompok"
    
    # JS / JOBSHEET / PERTEMUAN
    match = re.search(r"(JS|JOBSHEET|PERTEMUAN|SESI|MODUL|MODULE)\s*0?(\d+)", text)
    if match:
        return f"js{int(match.group(2)):02d}"

    # PRAKTIKUM
    match = re.search(r"PRAKTIKUM\s*0?(\d+)", text)
    if match:
        return f"js{int(match.group(1)):02d}"

    return None


# =====================================================
# IDENTIFIKASI FILE TYPE
# =====================================================

def normalize_filename(file):

    name = file.upper()
    ext = os.path.splitext(file)[1].lower()
    
    # PBL
    if "PBL" in name:
        return f"pbl{ext}"
    
    # KELOMPOK
    if re.search(r"KELOMPOK|GROUP", name):
        return f"kelompok{ext}"

    # UTS
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", name):
        return f"uts{ext}"

    # UAS
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", name):
        return f"uas{ext}"
    
    # KUIS
    if re.search(r"KUIS|QUIZ", name):
        return f"kuis{ext}"

    # PRAKTIKUM
    match = re.search(r"\bP\s*0?(\d+)", name)
    if match:
        return f"p{int(match.group(1)):02d}{ext}"

    # TUGAS PRAKTIKUM
    match = re.search(r"(TP|TG|TUGAS)\s*0?(\d+)", name)
    if match:
        return f"tp{int(match.group(2)):02d}{ext}"

    return None


# =====================================================
# HAPUS FOLDER YANG TIDAK PERLU
# =====================================================
def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)
    
def remove_unnecessary(repo, remove_list, nim):

    for root, dirs, files in os.walk(repo):

        for d in dirs[:]:
            if d in remove_list:
                path = os.path.join(root, d)

                try:
                    shutil.rmtree(path, onerror=remove_readonly)
                    dirs.remove(d)
                    write_log(nim, "REMOVE_FOLDER", path)
                except Exception as e:
                    write_log(nim, "REMOVE_FOLDER", path, "FAILED", str(e))
                    tqdm.write(f"[ERROR] {nim} | REMOVE_FOLDER | {path} | {str(e)}")


# =====================================================
# CARI MODULE DARI PATH
# =====================================================

def detect_module_from_path(root, file):

    module = None

    # cek semua folder dalam path
    for part in root.split(os.sep):

        module = identify_module(part)

        if module:
            return module

    # jika tidak ditemukan → cek dari file
    module = identify_module(file)

    return module


# =====================================================
# NORMALISASI REPOSITORY
# =====================================================

def normalize_repository(repo_path):

    nim = os.path.basename(repo_path)
    normalized_repo = os.path.join(normalized_folder, nim)

    if os.path.exists(normalized_repo):
        shutil.rmtree(normalized_repo)

    shutil.copytree(repo_path, normalized_repo)

    remove_list = [".env", ".venv", "env", "venv", "__pycache__", ".git"]
    remove_unnecessary(normalized_repo, remove_list, nim)

    unclassified = os.path.join(normalized_repo, "unclassified")
    os.makedirs(unclassified, exist_ok=True)

    moved_files = []

    # scan semua file dulu
    for root, dirs, files in os.walk(normalized_repo):
        # jangan masuk ke folder .git
        dirs[:] = [d for d in dirs if d != ".git"]
        for file in files:

            if not (file.endswith(".py") or file.endswith(".ipynb")):
                continue

            file_path = os.path.join(root, file)

            moved_files.append((root, file, file_path))


    # proses pemindahan
    for root, file, file_path in tqdm(
        moved_files,
        desc=f"{nim}",
        unit="file",
        leave=False,
        dynamic_ncols=True
    ):

        module = detect_module_from_path(root, file)

        if module:
            target_folder = os.path.join(normalized_repo, module)
        else:
            target_folder = unclassified

        os.makedirs(target_folder, exist_ok=True)

        new_name = normalize_filename(file)
        if new_name and new_name != file:
            write_log(nim, "RENAME_FILE", f"{file} -> {new_name}")

        if not new_name:
            new_name = file

        target_path = os.path.join(target_folder, new_name)

        # hindari overwrite
        counter = 1
        base, ext = os.path.splitext(new_name)

        while os.path.exists(target_path):

            target_path = os.path.join(
                target_folder,
                f"{base}_{counter}{ext}"
            )
            write_log(nim, "DUPLICATE_RENAME", f"{new_name} -> {base}_{counter}{ext}")
            counter += 1

        try:
            shutil.move(file_path, target_path)
            write_log(nim, "MOVE_FILE", f"{file_path} -> {target_path}")
        except Exception as e:
            write_log(nim, "MOVE_FILE", file_path, "FAILED", str(e))
            tqdm.write(f"[ERROR] {nim} | MOVE_FILE | {file_path} | {str(e)}")


# =====================================================
# NORMALISASI DATASET
# =====================================================

print("\nMulai normalisasi dataset...\n")

total_repo = 0
repos = [
    r for r in os.listdir(projects_folder)
    if os.path.isdir(os.path.join(projects_folder, r))
]

for repo in tqdm(repos, desc="Normalizing Repositories", unit="repo"):
    repo_path = os.path.join(projects_folder, repo)
    try:
        normalize_repository(repo_path)
    except Exception as e:
        tqdm.write(f"[ERROR] Repository {repo} gagal diproses: {str(e)}")
        write_log(repo, "NORMALIZE_REPO", repo_path, "FAILED", str(e))
    total_repo += 1


print("\n===== NORMALISASI SELESAI =====")
print(f"Total repository : {total_repo}")

# ===============================
# SIMPAN LOG
# ===============================

df_log = pd.DataFrame(log_records)

df_log.to_excel(normStruktur_excel, index=False)

print("\nLog normalisasi disimpan di:")
print(normStruktur_excel)

print("="*35)
print("DATA LOG NORMALISASI")
print("-"*35)
display(df_log.head(10))

normalisasi baru

In [ ]:
# CORE LOGIC: NORMALISASI REPOSITORY
# Tahap utama:
#   1. Membuat ulang folder tujuan di direktori NORM
#   2. Membersihkan folder tidak perlu
#   3. Mengklasifikasikan file .py / .ipynb ke subfolder modul
#   4. Menormalkan nama file ke format standar
#   5. Menangani duplikasi nama file secara otomatis
# ------------------------------------------------------------------------------

def normalize_repository(repo_path):
    nim = os.path.basename(repo_path)
    target_repo_root = os.path.join(DIRS['NORM'], nim)

    if os.path.exists(target_repo_root):
        shutil.rmtree(target_repo_root, onerror=remove_readonly)
    os.makedirs(target_repo_root, exist_ok=True)

    remove_list = [".env", ".venv", "env", "venv", "__pycache__", ".git", "colab lama"]
    remove_unnecessary(target_repo_root, remove_list, nim)
    
    unclassified_dir = os.path.join(target_repo_root, "unclassified")
    os.makedirs(unclassified_dir, exist_ok=True)

    files_to_move = []
    for root, dirs, files in os.walk(repo_path):
        dirs[:] = [d for d in dirs if d not in [".git", ".venv", "__pycache__", "env", "colab lama"]]
        for file in files:
            if file.endswith((".py", ".ipynb")):
                files_to_move.append((root, file, os.path.join(root, file)))

    for root, original_name, full_path in files_to_move:
        module = detect_module_from_path(root, original_name)
        target_subfolder = os.path.join(target_repo_root, module) if module else unclassified_dir
        os.makedirs(target_subfolder, exist_ok=True)

        new_name = normalize_filename(original_name) or original_name
        if new_name != original_name:
            write_log(nim, "RENAME_FILE", f"{original_name} -> {new_name}")

        target_file_path = os.path.join(target_subfolder, new_name)

        # Penanganan nama file duplikat
        if os.path.exists(target_file_path):
            base, ext = os.path.splitext(new_name)
            counter = 2
            while os.path.exists(os.path.join(target_subfolder, f"{base}{counter}{ext}")):
                counter += 1
            new_name = f"{base}{counter}{ext}"
            target_file_path = os.path.join(target_subfolder, new_name)
            write_log(nim, "DUPLICATE_HANDLE", f"Auto rename: {original_name} -> {new_name}")

        try:
            shutil.copy2(full_path, target_file_path)
            write_log(nim, "MOVE_FILE", f"{original_name} -> {module or 'unclassified'}")
        except Exception as e:
            write_log(nim, "MOVE_FILE", original_name, "FAILED", str(e))

# ------------------------------------------------------------------------------
# EKSEKUSI: JALANKAN NORMALISASI SELURUH DATASET
# Memproses semua repository dalam direktori RAW secara berurutan.

print("="*60)
print(f"{' PROSES NORMALISASI DIREKTORI ' : ^60}")
print("-"*60)

# Ambil daftar semua subfolder (repository) dari direktori RAW
raw_repos = [
    r for r in os.listdir(DIRS['RAW']) 
    if os.path.isdir(os.path.join(DIRS['RAW'], r))
]

total_processed = 0
log_records = [] # Reset log sebelum eksekusi dimulai

for repo in tqdm(raw_repos, desc="Normalizing Repositories", unit="repo"):
    repo_path = os.path.join(DIRS['RAW'], repo)
    try:
        normalize_repository(repo_path)
        total_processed += 1
    except Exception as e:
        tqdm.write(f"[ERROR] Gagal proses {repo}: {str(e)}")
        write_log(repo, "NORMALIZE_REPO", "Global Error", "FAILED", str(e))

print("="*60)
print(f"\n[SELESAI] Berhasil memproses {total_processed} repository.")

# Data Cleaning


### Project Level (CProfile)


In [ ]:
# HELPER FUNCTIONS (PROJECT LEVEL CPROFILE)
# -----------------------------------------------------------------
TIMEOUT = 30
ITERATIONS = 3

# HITUNG JUMLAH FUNCTION PER FILE
def count_functions_in_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            tree = ast.parse(f.read())

        function_count = sum(
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            for node in ast.walk(tree)
        )
        return function_count

    except Exception:
        return None


# EKSTRAK METADATA FILE
def extract_metadata(file_path, dataset_folder):
    rel_path = os.path.relpath(file_path, dataset_folder)
    parts = rel_path.split(os.sep)

    file_name = parts[-1]
    nim = parts[0] if len(parts) >= 2 else "unknown"
    module = parts[1] if len(parts) >= 3 else "root"

    return nim, module, file_name


# JALANKAN CPROFILE
def run_with_cprofile(file_path):
    runtimes = []

    try:
        for _ in range(ITERATIONS):
            with tempfile.NamedTemporaryFile(delete=False) as temp_prof:
                profile_file = temp_prof.name

            subprocess.run(
                ["python", "-m", "cProfile", "-o", profile_file, file_path],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE,
                timeout=TIMEOUT
            )

            stats = pstats.Stats(profile_file)
            runtimes.append(stats.total_tt)

            os.remove(profile_file)

        runtime = sum(runtimes) / len(runtimes)
        function_count = count_functions_in_file(file_path)

        return "SUCCESS", runtime, function_count, ""

    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, None, "Execution timeout"

    except Exception as e:
        return "FAILED", None, None, str(e)

In [ ]:
# EXECUTION (PROJECT LEVEL CPROFILE)
# -----------------------------------------------------------------

DATASET_INPUT = DIRS["AUTOPEP8"]
FILE_REPORT = RESULTS["RUN_PROJECT"]

py_files = []

for root, dirs, files in os.walk(DATASET_INPUT):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for file in files:
        if file.endswith(".py"):
            py_files.append(os.path.join(root, file))

total_files = len(py_files)

print("=" * 60)
print(f"{'MULAI PROFILING PROJECT LEVEL (CPROFILE)':^60}")
print("-" * 60)

records = []

success_count = 0
fail_count = 0
timeout_count = 0

pbar = tqdm(
    py_files,
    total=total_files,
    desc="Profiling Python Files",
    unit="file"
)

for file_path in pbar:
    nim, module, file_name = extract_metadata(file_path, DATASET_INPUT)
    status, runtime, function_count, message = run_with_cprofile(file_path)

    if status == "SUCCESS":
        success_count += 1
    elif status == "FAILED":
        fail_count += 1
    elif status == "TIMEOUT":
        timeout_count += 1

    records.append({
        "nim": nim,
        "module": module,
        "file_name": file_name,
        "function_count": function_count,
        "status": status,
        "execution_time_seconds": runtime,
        "message": message
    })

    pbar.set_postfix({
        "success": success_count,
        "fail": fail_count,
        "timeout": timeout_count
    })

pbar.close()

df_cprofile = pd.DataFrame(records)

if not df_cprofile.empty:
    df_cprofile["execution_time_seconds"] = (
        df_cprofile["execution_time_seconds"].fillna(0).round(6)
    )

print(f"{' Selesai ':-^60}")
print(f"Total file diproses : {len(df_cprofile)}")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")
print("="*60)

          MULAI PROFILING PROJECT LEVEL (CPROFILE)          


Profiling Python Files:   0%|          | 0/164 [00:00<?, ?file/s]

=====================Profiling selesai======================
Total file diproses : 164
Diproses pada 2026-04-06 17:40:27


In [ ]:
# SUMMARY & REPORT
# ------------------------------------------------------------------------

# simpan ke excel
df_cprofile.to_excel(FILE_REPORT, index=False)

print("=" * 60)
print(f"{'RINGKASAN HASIL PROFILING':^60}")
print("-" * 60)
print(f"{'Total File Dianalisis':<30}: {len(df_cprofile)}")
print(f"{'File SUCCESS':<30}: {(df_cprofile['status'] == 'SUCCESS').sum()}")
print(f"{'File FAILED':<30}: {(df_cprofile['status'] == 'FAILED').sum()}")
print(f"{'File TIMEOUT':<30}: {(df_cprofile['status'] == 'TIMEOUT').sum()}")
print(f"{'Rata-rata Runtime':<30}: {round(df_cprofile['execution_time_seconds'].mean(), 6)}\n")

print("-" * 60)
print(f"{'File Report disimpan di ':<30}: {FILE_REPORT}\n")
display(df_cprofile.head(10))

print("=" * 60)
# TOP 10 RUNTIME
df_success = df_cprofile[df_cprofile["status"] == "SUCCESS"].copy()

print(f"{'10 RUNTIME TERLAMA':^60}")
top_slowest = (
    df_success
    .sort_values("execution_time_seconds", ascending=False)
    [["nim", "module", "file_name", "function_count", "execution_time_seconds"]]
    .head(10)
)
display(top_slowest)
print("=" * 60)
print(f"{'10 RUNTIME TERCEPAT':^60}")
top_fastest = (
    df_success
    .sort_values("execution_time_seconds", ascending=True)
    [["nim", "module", "file_name", "function_count", "execution_time_seconds"]]
    .head(10)
)
display(top_fastest)

print("-" * 60)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                 RINGKASAN HASIL PROFILING                  
------------------------------------------------------------
Total File Dianalisis         : 164
File SUCCESS                  : 164
File FAILED                   : 0
File TIMEOUT                  : 0
Rata-rata Runtime             : 0.628475
------------------------------------------------------------
File Report disimpan di       : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\Output(6)\07a_runtime_project_level.xlsx
                       PREVIEW REPORT                       


,nim,module,file_name,function_count,status,execution_time_seconds,message
0,2241720092,js02,p01.py,0,SUCCESS,0.485991,
1,2241720092,js02,p02.py,0,SUCCESS,0.251245,
2,2241720092,js02,p03.py,0,SUCCESS,3.383076,
3,2241720092,js02,p04.py,0,SUCCESS,0.001265,
4,2241720092,js02,tp.py,0,SUCCESS,0.222819,
5,2241720092,js03,p01.py,0,SUCCESS,0.000949,
6,2241720092,js03,p02.py,0,SUCCESS,0.001017,
7,2241720092,js03,p03.py,0,SUCCESS,0.197871,
8,2241720092,js03,p04.py,0,SUCCESS,0.085754,
9,2241720092,js03,tp.py,0,SUCCESS,0.178364,


------------------------------------------------------------
                     10 RUNTIME TERLAMA                     


,nim,module,file_name,function_count,execution_time_seconds
2,2241720092,js02,p03.py,0,3.383076
157,2341720217,js13,p02.py,0,2.250202
73,2341720176,js05,tp.py,0,2.057018
78,2341720176,js07,p04.py,0,1.547714
77,2341720176,js07,p01.py,0,1.532111
100,2341720191,js03,p02.py,0,1.496484
49,2341720095,js13,p02.py,0,1.493333
123,2341720191,js13,p02.py,0,1.459661
60,2341720176,js02,p03.py,0,1.421862
115,2341720191,js07,p06.py,0,1.419750


------------------------------------------------------------
                    10 RUNTIME TERCEPAT                     


,nim,module,file_name,function_count,execution_time_seconds
55,2341720163,unclassified,tp.py,0,0.000006
56,2341720176,js01,p01.py,0,0.000006
128,2341720217,js01,DEMO_P1_JS01.py,0,0.000007
129,2341720217,js01,tp.py,0,0.000008
62,2341720176,js02,tp.py,0,0.000009
161,2341720217,js14,p02.py,0,0.000686
126,2341720191,js14,p02.py,0,0.000694
125,2341720191,js14,p01.py,0,0.000714
79,2341720176,js07,p06.py,0,0.000723
113,2341720191,js07,p03.py,0,0.000730


------------------------------------------------------------
Diproses pada 2026-04-06 17:40:27


In [ ]:
def normalisasi_whitespace(kode):
    """
    Normalisasi whitespace yang aman:
    1) hapus trailing spaces
    2) batasi blank line berurutan maksimal 1
    """
    lines = kode.split('\n')
    lines_bersih = []

    blank_streak = 0

    for line in lines:
        # hapus trailing spaces/tab di akhir baris
        stripped = line.rstrip()

        # cek blank line
        if stripped == '':
            blank_streak += 1

            # simpan hanya 1 blank line berturut-turut
            if blank_streak <= 1:
                lines_bersih.append('')
        else:
            blank_streak = 0
            lines_bersih.append(stripped)

    # hapus blank line di awal file
    while lines_bersih and lines_bersih[0] == '':
        lines_bersih.pop(0)

    # hapus blank line di akhir file
    while lines_bersih and lines_bersih[-1] == '':
        lines_bersih.pop()

    return '\n'.join(lines_bersih)

In [ ]:
# HELPER FUNCTION (CLEANING)
# ------------------------------------------------------------------------------

# HAPUS KOMENTAR
# Tujuan: Menghapus semua komentar (#) dari file .py di converted_py.
# Metode: Tokenize → Fallback regex jika gagal
def hapus_komentar_fallback(kode):
    lines_bersih = []
    for line in kode.split('\n'):
        if '#' in line:
            in_string = False
            chars = list(line)
            for i, ch in enumerate(chars):
                if ch in ('"', "'") and (i == 0 or chars[i-1] != '\\'):
                    in_string = not in_string
                if ch == '#' and not in_string:
                    line = line[:i]
                    break
        lines_bersih.append(line.rstrip())
    return '\n'.join(lines_bersih)

def hapus_komentar(kode):
    try:
        tokens = tokenize.generate_tokens(io.StringIO(kode).readline)
        tokens_bersih = [t for t in tokens if t.type != tokenize.COMMENT]
        return tokenize.untokenize(tokens_bersih), 'TOKENIZE'
    except:
        return hapus_komentar_fallback(kode), 'FALLBACK'


# HAPUS DOCSTRING
class DocstringRemover(ast.NodeTransformer):
    def _strip(self, node):
        if (node.body and isinstance(node.body[0], ast.Expr) and
            isinstance(node.body[0].value, ast.Constant) and
                isinstance(node.body[0].value.value, str)):
            node.body.pop(0)
        if not node.body:
            node.body.append(ast.Pass())
        self.generic_visit(node)
        return node
    visit_Module = visit_FunctionDef = visit_AsyncFunctionDef = visit_ClassDef = _strip

def hapus_docstring(kode):
    try:
        tree = ast.parse(kode)
        tree = DocstringRemover().visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree), 'AST'
    except:
        # Fallback sederhana jika AST gagal
        lines = kode.split('\n')
        return '\n'.join([l for l in lines if not l.strip().startswith(('"""', "'''"))]), 'FALLBACK'


# NORMALISASI WHITESPACE
def normalisasi_whitespace(kode):
    lines = kode.split('\n')
    # Hapus trailing spaces dan baris yang benar-benar kosong
    lines_bersih = [l.rstrip() for l in lines if l.strip() != '']
    return '\n'.join(lines_bersih)

In [ ]:
# EKSEKUSI
# ------------------------------------------------------------------------------

cleaning_logs = []

# Pastikan folder target CLEAN sudah ada
os.makedirs(DIRS['CLEAN'], exist_ok=True)

# Kumpulkan semua file .py dari folder CONV
all_py_files = []
for root, dirs, files in os.walk(DIRS['CONV']):
    for fname in files:
        if fname.endswith('.py'):
            all_py_files.append((os.path.join(root, fname), root))

print(f"{' MULAI CLEANING CODE ':=^60}")

for source_path, root in tqdm(all_py_files, desc="Cleaning Process", unit="file"):
    # Buat path tujuan di folder CLEAN
    rel_path = os.path.relpath(root, DIRS['CONV'])
    target_dir = os.path.join(DIRS['CLEAN'], rel_path)
    os.makedirs(target_dir, exist_ok=True)

    target_path = os.path.join(target_dir, os.path.basename(source_path))
    nim = rel_path.split(os.sep)[0] if rel_path != "." else "unknown"

    try:
        with open(source_path, 'r', encoding='utf-8', errors='replace') as f:
            kode = f.read()

        # Jalankan urutan pembersihan
        kode, m_komentar = hapus_komentar(kode)
        kode, m_docstring = hapus_docstring(kode)
        kode = normalisasi_whitespace(kode)

        # Simpan ke lokasi baru (DIRS['CLEAN'])
        with open(target_path, 'w', encoding='utf-8') as f:
            f.write(kode)

        cleaning_logs.append({
            "nim": nim, "file": os.path.basename(source_path),
            "method_komentar": m_komentar, "method_docstring": m_docstring,
            "status": "SUCCESS"
        })
    except Exception as e:
        cleaning_logs.append({
            "nim": nim, "file": os.path.basename(source_path),
            "status": "FAILED", "error": str(e)
        })

print(f"\n{' PROSES BERHASIL DISIMPAN DI FOLDER CLEAN ' : =^60}")

=================== MULAI CLEANING CODE ====================


Cleaning Process:   0%|          | 0/219 [00:00<?, ?file/s]


          PROSES BERHASIL DISIMPAN DI FOLDER CLEAN          


In [ ]:
# Simpan Log ke Excel khusus cleaning
df_clean = pd.DataFrame(cleaning_logs)
df_clean.to_excel(RESULTS['CLEAN_REPORT'], index=False)

print("\n" + "="*60)
print(f"{' STATISTIK DATASET CLEAN ' : ^60}")
print("="*60)
print(f"{'Lokasi Input (CONV)':<30} : {DIRS['CONV']}")
print(f"{'Lokasi Output (CLEAN)':<30} : {DIRS['CLEAN']}")
print(
    f"{'Total File Selesai':<30} : {len(df_clean[df_clean['status']=='SUCCESS']):<5}")
print(f"{'Laporan':<30} : {os.path.basename(RESULTS['CLEAN_REPORT'])}")
print("-" * 60)

if not df_clean.empty:
    print("[INFO] Ringkasan Metode:")
    if 'method_komentar' in df_clean.columns:
        print(df_clean['method_komentar'].value_counts())

print("="*60)
display(df_clean.head(10))


                STATISTIK DATASET CLEAN                
Lokasi Input (CONV)            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset6\03_Converted
Lokasi Output (CLEAN)          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset6\04_Cleaned
Total File Selesai             :   219
Laporan                        : 04_cleaning_report.xlsx
-------------------------------------------------------

[INFO] Ringkasan Metode:
TOKENIZE    219
Name: method_komentar, dtype: int64


,nim,file,method_komentar,method_docstring,status
0,2241720092,p01.py,TOKENIZE,AST,SUCCESS
1,2241720092,p02.py,TOKENIZE,AST,SUCCESS
2,2241720092,p03.py,TOKENIZE,AST,SUCCESS
3,2241720092,p04.py,TOKENIZE,AST,SUCCESS
4,2241720092,tp.py,TOKENIZE,AST,SUCCESS
5,2241720092,p01.py,TOKENIZE,AST,SUCCESS
6,2241720092,p02.py,TOKENIZE,AST,SUCCESS
7,2241720092,p03.py,TOKENIZE,AST,SUCCESS
8,2241720092,p04.py,TOKENIZE,AST,SUCCESS
9,2241720092,tp.py,TOKENIZE,AST,SUCCESS
